# Customer Churn Prediction — End-to-End Machine Learning

**Python · pandas · scikit-learn · EDA · Classification · Model Evaluation**

**Business question:** Which customers are more likely to churn, and which observable characteristics are associated with that risk?

**Data:** IBM Telco Customer Churn sample (7,043 customers, 21 original columns). This is a demonstration dataset, not Helios Towers data.

**Methodology:** inspect → clean → explore → stratified split → leakage-safe preprocessing → compare models → evaluate → interpret → save.

> Run cells in order. Metrics and charts are computed from the dataset; no performance numbers are assumed in advance.

## 0. Setup
If needed in Colab, uncomment the installation cell.

In [ ]:
# %pip install -q pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, average_precision_score, RocCurveDisplay,
                             PrecisionRecallDisplay, precision_recall_curve)
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
print("pandas:", pd.__version__)

## 1. Load the dataset
The notebook first looks for `data/Telco-Customer-Churn.csv`, then tries the public IBM raw CSV URL. If offline, place the file in `data/`.

In [ ]:
LOCAL_PATH = Path("data/Telco-Customer-Churn.csv")
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
if LOCAL_PATH.exists():
    df = pd.read_csv(LOCAL_PATH)
else:
    try:
        df = pd.read_csv(DATA_URL)
    except Exception as exc:
        raise RuntimeError("Dataset unavailable. Download the IBM CSV and save it to data/Telco-Customer-Churn.csv") from exc
print("Shape:", df.shape)
display(df.head())

## 2. Data understanding and quality checks

In [ ]:
df.info()
display(df.isna().sum().rename("null_count").to_frame())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())
display(df["Churn"].value_counts(dropna=False).to_frame("count"))
display((df["Churn"].value_counts(normalize=True) * 100).round(2).to_frame("percent"))

In [ ]:
# Strings consisting of spaces are not always detected by isna().
total_charges_as_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
invalid_total = total_charges_as_numeric.isna()
print("Unparseable TotalCharges:", invalid_total.sum())
display(df.loc[invalid_total, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]])
print("All affected customers have tenure=0:", bool(df.loc[invalid_total, "tenure"].eq(0).all()))

## 3. Minimal data cleaning (before splitting)

- Convert `TotalCharges` to numeric and preserve an indicator for originally missing values.
- **Do not** fit encoders, scalers, or statistical imputers on the entire dataset.
- We will use a zero fill for `TotalCharges` *only if* all its missing values have `tenure == 0`. This is a documented business assumption, not a verified billing fact.
- `customerID` is an identifier, not a predictive feature.

In [ ]:
data = df.copy()
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data["TotalCharges_missing"] = data["TotalCharges"].isna().astype(int)
assert data["Churn"].isin(["Yes", "No"]).all(), "Unexpected target label"
assert data["customerID"].is_unique, "Duplicate customer identifiers need investigation"
missing_total = data["TotalCharges"].isna()
assert data.loc[missing_total, "tenure"].eq(0).all(), "Missing TotalCharges not limited to tenure=0; revisit strategy"
data.loc[missing_total, "TotalCharges"] = 0.0
print("Remaining nulls:", data.isna().sum().sum())
print("Shape:", data.shape)
display(data[["tenure", "MonthlyCharges", "TotalCharges"]].describe())

## 4. Exploratory Data Analysis (EDA)
These plots describe associations, not causal effects.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=data, x="Churn", order=["No", "Yes"], ax=ax)
ax.set(title="Customer churn distribution", xlabel="Churn", ylabel="Customers")
plt.show()

In [ ]:
contract_summary = (data.groupby("Contract", observed=True)["Churn"]
                    .agg(total_customers="count", churned_customers=lambda s: s.eq("Yes").sum()))
contract_summary["churn_rate_pct"] = (100 * contract_summary["churned_customers"] / contract_summary["total_customers"])
display(contract_summary.sort_values("churn_rate_pct", ascending=False).round(2))
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=contract_summary.reset_index(), x="Contract", y="churn_rate_pct", ax=ax)
ax.set(title="Churn rate by contract type", ylabel="Churn rate (%)", xlabel="Contract")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=data, x="Churn", y="tenure", order=["No", "Yes"], ax=axes[0])
axes[0].set(title="Tenure by churn")
sns.boxplot(data=data, x="Churn", y="MonthlyCharges", order=["No", "Yes"], ax=axes[1])
axes[1].set(title="Monthly charges by churn")
plt.tight_layout()
plt.show()

In [ ]:
for feature in ["InternetService", "PaymentMethod", "TechSupport"]:
    rates = (data.groupby(feature, observed=True)["Churn"]
             .apply(lambda s: 100 * s.eq("Yes").mean())
             .sort_values(ascending=False))
    print(f"\n{feature}: churn rate (%)")
    display(rates.round(2).to_frame("churn_rate_pct"))

### EDA reflection

After running the charts, write down:
1. Which contract type has the highest observed churn rate?
2. How do tenure and monthly charges differ between churners and non-churners?
3. What *other factors* could explain these associations?

Avoid statements such as "contract type causes churn" without a causal study.

## 5. Define features and create a stratified holdout test set
The test set is reserved for the final evaluation. No preprocessing model is fitted before this split.

In [ ]:
X = data.drop(columns=["Churn", "customerID"])
y = data["Churn"].map({"No": 0, "Yes": 1}).astype(int)
assert y.notna().all()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train churn rate:", round(y_train.mean(), 4))
print("Test churn rate:", round(y_test.mean(), 4))

## 6. Leakage-safe preprocessing and model pipelines
We fit preprocessing *inside* each training fold using scikit-learn pipelines.

In [ ]:
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
print("Categorical:", categorical_features)
print("Numeric:", numeric_features)

# Support both older and newer scikit-learn releases.
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", encoder)
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

models = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(n_estimators=250, min_samples_leaf=3,
                                          class_weight="balanced", random_state=RANDOM_STATE,
                                          n_jobs=-1))
    ])
}

## 7. Cross-validation on training data only
Compare ROC-AUC, average precision (PR-AUC), recall, and F1. Cross-validation estimates are not test-set results.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "average_precision": "average_precision", "recall": "recall", "f1": "f1"}
rows = []
for name, pipe in models.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    row = {"model": name}
    for metric in scoring:
        row[f"cv_{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"cv_{metric}_std"] = scores[f"test_{metric}"].std()
    rows.append(row)
cv_results = pd.DataFrame(rows).sort_values("cv_roc_auc_mean", ascending=False)
display(cv_results.round(3))

## 8. Fit selected model and evaluate once on held-out test data
The selection is based on training-fold ROC-AUC. Threshold 0.50 is a baseline, not necessarily optimal for retention costs.

In [ ]:
best_name = cv_results.iloc[0]["model"]
best_pipeline = models[best_name]
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]
print("Selected model:", best_name)
print("Test ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("Test average precision:", round(average_precision_score(y_test, y_proba), 4))
print("\nClassification report (threshold=0.50):\n")
print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"], zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["No", "Yes"], ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[2])
axes[2].set_title("Precision–Recall curve")
plt.tight_layout()
plt.show()

## 9. Interpret the model (permutation importance)

Permutation importance is computed on the holdout set for *exploratory interpretation only*. Do not repeatedly change the model based on test-set importance. Feature importance indicates association with predictions, not causation.

In [ ]:
importance = permutation_importance(
    best_pipeline, X_test, y_test, scoring="roc_auc",
    n_repeats=5, random_state=RANDOM_STATE, n_jobs=1
)
feature_importance = (pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std
}).sort_values("importance_mean", ascending=False))
display(feature_importance.head(12).round(4))
plt.figure(figsize=(9, 5))
sns.barplot(data=feature_importance.head(10), x="importance_mean", y="feature", color="steelblue")
plt.title("Top permutation importances (test ROC-AUC decrease)")
plt.xlabel("Mean ROC-AUC decrease after permutation")
plt.tight_layout()
plt.show()

## 10. Save the fitted pipeline

The saved artifact contains the preprocessing steps and trained estimator. Only load `.joblib` files from trusted sources. For production, pin package versions and validate inputs.

In [ ]:
output_dir = Path("models")
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "customer_churn_pipeline.joblib"
joblib.dump(best_pipeline, model_path)
print("Saved model to:", model_path)

# Example prediction on one held-out customer (features only)
example = X_test.iloc[[0]]
print("Example churn probability:", round(float(best_pipeline.predict_proba(example)[0, 1]), 4))

## 11. Conclusions and next steps

Fill this section *after executing the notebook*:

- **Highest churn rate by contract:** [observed value]
- **Most relevant EDA patterns:** [observations]
- **Selected model:** [model]
- **Cross-validated ROC-AUC:** [mean ± std]
- **Holdout ROC-AUC / PR-AUC:** [results]
- **Churn recall / precision:** [results]
- **Limitations:** Demonstration dataset; observational relationships; no causal inference; potential drift and fairness issues in real deployment.

**Next:** document reproducible metrics in the README; refactor preprocessing/training into `src/`; compare with PySpark MLlib; optionally build a Streamlit demonstration with public sample data only.

**Data source:** [IBM Telco Customer Churn GitHub repository](https://github.com/IBM/telco-customer-churn-on-icp4d).